In [33]:
import time, json, shutil
import numpy as np, pandas as pd, torch, torch.nn as nn
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import mobilenet_v3_small, MobileNet_V3_Small_Weights

EPILL = Path("../data/raw/epillid/ePillID_data")
MODELS = Path("../models")
MODELS.mkdir(exist_ok=True)
device = "cuda" if torch.cuda.is_available() else "cpu"

df = pd.read_csv(EPILL / "all_labels.csv")
cons = df[~df.is_ref]
per_type = cons.pilltype_id.value_counts()

def data_dir(N):
    return Path("../data/processed") if N == 40 else Path(f"../data/processed_{N}")

def model_path(N):
    return MODELS / "best_classifier.pt" if N == 40 else MODELS / f"classifier_{N}.pt"

def score_path(N):
    return MODELS / "best_score.json" if N == 40 else MODELS / f"score_{N}.json"

In [34]:
current40 = pd.read_csv("../data/processed/catalog.csv").pilltype_id.tolist()

rest = (per_type[per_type >= 4]
        .index
        .difference(current40)
        .to_series()
        .sample(frac=1, random_state=42)
        .tolist())

ORDER = current40 + rest
SIZES = [40, 100,250, len(ORDER)]

def make_split(N):
    OUT = data_dir(N)
    OUT.mkdir(parents=True, exist_ok=True)
    catalog = ORDER[:N]
    pd.DataFrame({"pilltype_id": catalog}).to_csv(OUT / "catalog.csv", index=False)

    c = cons[cons.pilltype_id.isin(catalog)].sample(frac=1, random_state=42).copy()
    c["rank"] = c.groupby("pilltype_id").cumcount()
    c["split"] = c["rank"].map(lambda r: "test" if r == 0 else ("val" if r == 1 else "train"))

    r = df[df.is_ref & df.pilltype_id.isin(catalog)].copy()
    r["split"] = "train"
    allp = pd.concat([c.drop(columns="rank"), r])
    assert not allp.image_path.duplicated().any()
    for name in ["train", "val", "test"]:
        allp[allp.split == name].to_csv(OUT / f"{name}.csv", index=False)
    print(N, allp.split.value_counts().to_dict())

for N in SIZES[1:]:
    make_split(N)

100 {'train': 470, 'test': 100, 'val': 100}
250 {'train': 1166, 'test': 250, 'val': 250}
648 {'train': 3017, 'test': 648, 'val': 648}


In [35]:
class PillDataset(Dataset):
    def __init__(self, csv_path, pills, transform):
        self.df = pd.read_csv(csv_path).reset_index(drop=True)
        self.transform = transform
        self.to_idx = {p: i for i, p in enumerate(pills)}

    def __len__(self):
        return len(self.df)

    def __getitem__(self, i):
        r = self.df.iloc[i]
        img = Image.open(EPILL / "classification_data" / r.image_path).convert("RGB")
        return self.transform(img), self.to_idx[r.pilltype_id]

norm = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    norm])

train_tf = transforms.Compose([
    transforms.RandomResizedCrop(224, scale=(0.7, 1.0)),
    transforms.RandomRotation(30),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    norm])

def evaluate(model, dl):
    model.eval()
    top1 = top5 = n = 0
    with torch.no_grad():
        for x, y in dl:
            idx = model(x.to(device)).cpu().topk(5, dim=1).indices
            top1 += (idx[:, 0] == y).sum().item()
            top5 += (idx == y[:, None]).any(dim=1).sum().item()
            n += len(y)
    return top1 / n, top5 / n

def build_model(n_pills):
    m = mobilenet_v3_small(weights=MobileNet_V3_Small_Weights.DEFAULT)
    m.classifier[3] = nn.Linear(1024, n_pills)
    return m.to(device)

def get_pills(N):
    return sorted(pd.read_csv(data_dir(N) / "catalog.csv").pilltype_id)

def get_loader(N, split, tf, shuffle=False, batch_size=16):
    ds = PillDataset(data_dir(N) / f"{split}.csv", get_pills(N), tf)
    return DataLoader(ds, batch_size=batch_size, shuffle=shuffle, num_workers=0)

In [36]:
def train_and_update(N, epochs=20, batch_size=16, lr=3e-4):
    pills = get_pills(N)
    save = model_path(N)
    score_file = score_path(N)

    tdl = get_loader(N, "train", train_tf, shuffle=True, batch_size=batch_size)
    vdl = get_loader(N, "val", preprocess, batch_size=batch_size)

    if score_file.exists():
        s = json.load(open(score_file))
        saved = (s["top1"], s["top5"])
    elif save.exists():
        old = build_model(len(pills))
        old.load_state_dict(torch.load(save, map_location=device))
        saved = evaluate(old, vdl)
    else:
        saved = (0, 0)
    print(f"N={N} saved model val (top-1, top-5): {saved}")

    torch.manual_seed(int(time.time()))
    m = build_model(len(pills))
    loss_fn = nn.CrossEntropyLoss()
    opt = torch.optim.AdamW(m.parameters(), lr=lr)

    for ep in range(epochs):
        t0 = time.time()
        m.train()
        total = 0
        for x, y in tdl:
            x, y = x.to(device), y.to(device)
            opt.zero_grad()
            loss = loss_fn(m(x), y)
            loss.backward()
            opt.step()
            total += loss.item() * len(y)
        v1, v5 = evaluate(m, vdl)
        print(f"N={N} epoch {ep+1:2d}  loss {total/len(tdl.dataset):.3f}  val top-1 {v1:.3f}  top-5 {v5:.3f}  ({time.time()-t0:.0f}s)")

    new = (v1, v5)
    print(f"N={N} new model val: {new}  |  saved model val: {saved}")
    if new > saved:
        if save.exists():
            shutil.copy(save, save.with_name(save.stem + "_backup.pt"))
        torch.save(m.state_dict(), save)
        json.dump({"top1": v1, "top5": v5}, open(score_file, "w"))
        print(f"N={N} overwritten with the new model")
    else:
        print(f"N={N} kept the old model")

In [37]:
old_model = MODELS / "best_classifier.pt"
if old_model.exists() and not (MODELS / "classifier_40.pt").exists():
    shutil.copy(old_model, MODELS / "classifier_40.pt")
    json.dump({"val1": 0.825, "val5": 0.925}, open(MODELS / "score_40.json", "w"))

In [38]:
train_and_update(100)
train_and_update(250)
train_and_update(SIZES[-1])

N=100 saved model val (top-1, top-5): (0.68, 0.91)
N=100 epoch  1  loss 4.554  val top-1 0.080  top-5 0.180  (3s)
N=100 epoch  2  loss 3.831  val top-1 0.170  top-5 0.400  (2s)
N=100 epoch  3  loss 2.894  val top-1 0.340  top-5 0.670  (2s)
N=100 epoch  4  loss 2.093  val top-1 0.410  top-5 0.790  (2s)
N=100 epoch  5  loss 1.503  val top-1 0.520  top-5 0.800  (3s)
N=100 epoch  6  loss 1.204  val top-1 0.560  top-5 0.810  (3s)
N=100 epoch  7  loss 0.884  val top-1 0.550  top-5 0.820  (2s)
N=100 epoch  8  loss 0.723  val top-1 0.560  top-5 0.850  (2s)
N=100 epoch  9  loss 0.563  val top-1 0.620  top-5 0.840  (2s)
N=100 epoch 10  loss 0.495  val top-1 0.590  top-5 0.820  (2s)
N=100 epoch 11  loss 0.437  val top-1 0.560  top-5 0.770  (2s)
N=100 epoch 12  loss 0.344  val top-1 0.650  top-5 0.850  (3s)
N=100 epoch 13  loss 0.283  val top-1 0.660  top-5 0.890  (3s)
N=100 epoch 14  loss 0.256  val top-1 0.650  top-5 0.860  (5s)
N=100 epoch 15  loss 0.224  val top-1 0.670  top-5 0.870  (7s)
N=10

In [39]:
rows = []
for N in SIZES:
    pills = get_pills(N)
    m = build_model(len(pills))
    m.load_state_dict(torch.load(model_path(N), map_location=device))

    own = evaluate(m, get_loader(N, "test", preprocess))

    common_ds = PillDataset(data_dir(40) / "test.csv", pills, preprocess)
    common = evaluate(m, DataLoader(common_ds, batch_size=16, num_workers=0))

    rows.append(dict(catalog=N, test_top1=own[0], test_top5=own[1],
                     same40_top1=common[0], same40_top5=common[1]))

res = pd.DataFrame(rows)
print(res.round(3))
res.to_csv(MODELS / "comparison.csv", index=False)  

   catalog  test_top1  test_top5  same40_top1  same40_top5
0       40      0.725      0.850        0.725        0.850
1      100      0.630      0.830        0.850        0.925
2      250      0.632      0.888        0.725        0.825
3      648      0.574      0.819        0.700        0.775
